# Q-Mini Add/Sub 2M — One-click English training

**Run the single code cell below.** It first asks Google Colab to connect Google Drive, then performs dataset preparation, tokenizer creation, the 2M-model preflight, English pretraining, human-authored assistant SFT, held-out evaluation, readiness checks and a packed-weight export.

The model is capped at **2,000,000 parameters** (1,985,684 with a 1,024-token vocabulary). The pipeline is CPU-only with two PyTorch threads. It uses hash-pinned Simple English Wikipedia, OASST1 and Dolly data; synthetic training records are excluded.

**License approval:** before downloading new data, the cell displays the CC BY-SA 3.0 and Apache-2.0 source terms and asks you to type `I ACCEPT`. Enter `I ACCEPT` once before the first download; extra explanatory words after it are accepted. The approval is saved in Drive and reused if a download is interrupted. If the Drive data already exists and its manifest is valid, it is reused.

**Resume:** data, tokenizer and checkpoints are stored in `MyDrive/Q mini 2 light/trainability_addsub_2m_v1`. If Colab disconnects, run this same cell again; it resumes from the newest compatible checkpoint. A free Colab CPU session may not finish 20,000 pretraining steps and 2,000 SFT steps in one connection.

**Important limitation:** `.qma` is a packed-weight archive, not the complete multiply-free integer mobile runtime. The held-out readiness report is an engineering check, not proof of factuality or reasoning.

In [ ]:
# Q-MINI ADD/SUB 2M — ONE-CELL ENGLISH TRAINING
# Run this cell once. It mounts Google Drive, prepares pinned English data,
# builds the 2M-capped model, pretrains, fine-tunes, evaluates and exports.
# Re-run the same cell after a Colab disconnect: Drive checkpoints are resumed.

from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import torch

REPO_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Q-mini-light-0.7M")

# This training route deliberately stays CPU-only.
DEVICE = "cpu"
CPU_THREADS = 2
PRETRAIN_STEPS = 20_000
PRETRAIN_BATCH_SIZE = 1
PRETRAIN_SEQUENCE_LENGTH = 512
PRETRAIN_SAVE_EVERY = 1_000
PRETRAIN_VALIDATE_EVERY = 1_000
SFT_STEPS = 2_000
SFT_BATCH_SIZE = 1
SFT_SAVE_EVERY = 250
MAX_PARAMETER_COUNT = 2_000_000

def run(command, *, cwd=None, env=None):
    print("\n>>>", " ".join(str(part) for part in command), flush=True)
    subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        check=True,
    )

print("\n[1/9] Getting the latest merged Q-Mini source...")
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Move/rename that temporary Colab folder, then rerun this cell."
        )
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], cwd=REPO_DIR)

REPO_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print("Repository commit:", REPO_COMMIT)

print("\n[2/9] Installing data/tokenizer dependencies (keeping Colab's PyTorch)...")
run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.24", "tokenizers>=0.15",
    "huggingface_hub>=0.28", "pyarrow>=15.0",
])
run([
    sys.executable, "-m", "pip", "install",
    "--no-deps", "-q", "-e", str(REPO_DIR),
])

# Do not rely only on the editable-install .pth file: Colab kernels can start
# with a sys.path that does not include the environment where pip registered it.
# Prefer the exact checkout used for this run and verify the import origin.
REPO_DIR = REPO_DIR.resolve()
PACKAGE_INIT = REPO_DIR / "qmini_addsub" / "__init__.py"
if not PACKAGE_INIT.is_file():
    raise FileNotFoundError(
        f"Q-Mini Add/Sub package is missing from the checked-out repository: {PACKAGE_INIT}"
    )
repo_import_path = str(REPO_DIR)
if repo_import_path in sys.path:
    sys.path.remove(repo_import_path)
sys.path.insert(0, repo_import_path)

import importlib
importlib.invalidate_caches()
import qmini_addsub
package_origin = Path(qmini_addsub.__file__).resolve()
if REPO_DIR not in package_origin.parents:
    raise RuntimeError(
        "Python imported qmini_addsub from outside the current repository checkout: "
        f"{package_origin}. Refusing to mix source versions."
    )
print("Python executable:", sys.executable)
print("Q-Mini package:", package_origin)
print("Repository commit:", REPO_COMMIT)

torch.set_num_threads(CPU_THREADS)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE, "(GPU is not used)")
print("CPU threads:", torch.get_num_threads())

from qmini_addsub.config import AddSubConfig, MAX_PARAMETER_BUDGET
from qmini_addsub.checkpoints import resolve_checkpoint
from qmini_addsub.train import sha256_file
from qmini_addsub.workspace import resolve_workspace_root

DRIVE_ROOT = Path("/content/drive/MyDrive/Q mini 2 light")
VERSION_ROOT = DRIVE_ROOT / "trainability_addsub_2m_v1"
WORKSPACE_ROOT = resolve_workspace_root(
    VERSION_ROOT,
    resume_existing=True,
)
DATA_DIR = WORKSPACE_ROOT / "data"
CACHE_DIR = WORKSPACE_ROOT / "cache"
ARTIFACT_DIR = WORKSPACE_ROOT / "artifacts"
PRETRAIN_DIR = WORKSPACE_ROOT / "runs" / "pretrain"
SFT_DIR = WORKSPACE_ROOT / "runs" / "sft"
EVALUATION_DIR = WORKSPACE_ROOT / "evaluation"
for folder in (DATA_DIR, CACHE_DIR, ARTIFACT_DIR, PRETRAIN_DIR, SFT_DIR, EVALUATION_DIR):
    folder.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = REPO_DIR / "configs" / "qmini_addsub_2m.json"
TOKENIZER_PATH = ARTIFACT_DIR / "tokenizer.json"
MODEL_CONFIG_PATH = ARTIFACT_DIR / "model_config.json"
MANIFEST_PATH = DATA_DIR / "dataset_manifest.json"
LICENSE_APPROVAL_PATH = WORKSPACE_ROOT / "dataset_license_approval.json"

print("\n[3/9] Checking the English human-authored dataset...")
if not MANIFEST_PATH.is_file():
    print("""
The pinned corpus uses:
  • Simple English Wikipedia — CC BY-SA 3.0
  • OpenAssistant OASST1 — Apache-2.0
  • Databricks Dolly 15k — CC BY-SA 3.0

The data preparation code pins source revisions and verifies source hashes.
It filters the data and creates separate train/validation/test splits.
Synthetic/model-generated training records are excluded.
Review the source cards and license terms before continuing.
""")
    # Accept the explicit approval phrase even when the user includes the
    # explanatory words shown in the previous notebook prompt.
    approval_record = None
    if LICENSE_APPROVAL_PATH.is_file():
        try:
            approval_record = json.loads(LICENSE_APPROVAL_PATH.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            approval_record = None
    approval_is_saved = (
        isinstance(approval_record, dict)
        and approval_record.get("format") == "qmini-source-license-approval-v1"
        and approval_record.get("accepted") is True
        and approval_record.get("sources") == [
            "simple_english_wikipedia_cc_by_sa_3",
            "openassistant_oasst1_apache_2",
            "databricks_dolly_15k_cc_by_sa_3",
        ]
    )
    if approval_is_saved:
        print("Reusing the source-license approval saved in Google Drive.")
    else:
        approval = input(
            "Enter I ACCEPT to approve the listed source licenses "
            "(additional explanatory words are okay): "
        ).strip()
        if not (approval == "I ACCEPT" or approval.startswith("I ACCEPT ")):
            raise RuntimeError(
                "Enter I ACCEPT to approve the listed dataset licenses. "
                "No dataset or model training was started."
            )
        approval_record = {
            "format": "qmini-source-license-approval-v1",
            "accepted": True,
            "approval_phrase": "I ACCEPT",
            "sources": [
                "simple_english_wikipedia_cc_by_sa_3",
                "openassistant_oasst1_apache_2",
                "databricks_dolly_15k_cc_by_sa_3",
            ],
        }
        approval_tmp = LICENSE_APPROVAL_PATH.with_suffix(".tmp")
        approval_tmp.write_text(
            json.dumps(approval_record, indent=2) + "\\n",
            encoding="utf-8",
        )
        approval_tmp.replace(LICENSE_APPROVAL_PATH)
        print("Source-license approval saved in Google Drive.")
    run([
        sys.executable, "scripts/prepare_qmini_addsub_data.py",
        "--output", DATA_DIR,
        "--cache", CACHE_DIR,
        "--accept-source-licenses",
    ], cwd=REPO_DIR)
else:
    print("Reusing the existing Drive dataset manifest; no data is overwritten.")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if (
    manifest.get("format") != "qmini-addsub-data-v1"
    or manifest.get("language") != "English"
    or manifest.get("synthetic_training_records_included") != 0
):
    raise RuntimeError(
        "Drive dataset manifest is not a verified English/non-synthetic Q-Mini Add/Sub dataset."
    )
for required_name in (
    "pretrain_train.txt", "pretrain_validation.txt", "pretrain_test.txt",
    "sft_train.jsonl", "sft_validation.jsonl", "sft_test.jsonl",
):
    if not (DATA_DIR / required_name).is_file():
        raise FileNotFoundError(f"Verified dataset is incomplete: {DATA_DIR / required_name}")
print("Dataset is verified. Synthetic training records:", manifest["synthetic_training_records_included"])
print("Instruction sources:", manifest.get("approved_instruction_sources"))

print("\n[4/9] Building/checking the tokenizer and model profile...")
if TOKENIZER_PATH.exists() != MODEL_CONFIG_PATH.exists():
    raise FileExistsError(
        "Only one of tokenizer.json/model_config.json exists. "
        "Keeping both untouched; inspect this Drive workspace before continuing."
    )
if not TOKENIZER_PATH.exists():
    run([
        sys.executable, "scripts/train_qmini_addsub_tokenizer.py",
        "--data-dir", DATA_DIR,
        "--output", TOKENIZER_PATH,
        "--vocab-size", "1024",
        "--config-template", PROFILE_PATH,
    ], cwd=REPO_DIR)
else:
    print("Reusing the tokenizer/profile already saved in Drive.")

model_config = AddSubConfig.from_dict(
    json.loads(MODEL_CONFIG_PATH.read_text(encoding="utf-8"))
)
parameter_count = model_config.estimated_parameter_count()
if parameter_count > MAX_PARAMETER_BUDGET:
    raise RuntimeError(
        f"Refusing to train {parameter_count:,} parameters; cap is {MAX_PARAMETER_BUDGET:,}."
    )
if (
    model_config.d_model != 128
    or model_config.d_hidden != 448
    or model_config.n_layers != 4
    or model_config.n_experts != 2
    or model_config.max_context != 512
):
    raise RuntimeError("The saved tokenizer is not paired with the intended 2M Add/Sub profile.")
print("Actual vocabulary:", model_config.vocab_size)
print("Exact trainable parameters:", f"{parameter_count:,}")
print("Absolute parameter ceiling:", f"{MAX_PARAMETER_BUDGET:,}")

token_streams = manifest.get("token_streams", {})
tokenizer_sha = sha256_file(TOKENIZER_PATH)
streams_ready = (
    manifest.get("tokenizer", {}).get("sha256") == tokenizer_sha
    and all(
        (DATA_DIR / f"pretrain_{split}.u16").is_file()
        and token_streams.get(split, {}).get("sha256")
            == sha256_file(DATA_DIR / f"pretrain_{split}.u16")
        for split in ("train", "validation", "test")
    )
)
if not streams_ready:
    print("Encoding/rebuilding hash-bound uint16 token streams...")
    run([
        sys.executable, "scripts/encode_qmini_addsub_streams.py",
        "--data-dir", DATA_DIR,
        "--tokenizer", TOKENIZER_PATH,
    ], cwd=REPO_DIR)
else:
    print("Verified token streams already exist; reusing them.")

print("\n[5/9] Running the full 2M-model training-engine preflight...")
run([
    sys.executable, "-m", "qmini_addsub.preflight",
    "--config", PROFILE_PATH,
    "--device", DEVICE,
    "--threads", str(CPU_THREADS),
    "--batch-size", "1",
    "--sequence-length", "16",
], cwd=REPO_DIR)

print("\n[6/9] English language pretraining...")
pretrain_checkpoint, pretrain_step = resolve_checkpoint(
    PRETRAIN_DIR,
    prefix="checkpoint",
    metrics_filename="metrics.jsonl",
    explicit=None,
)
if pretrain_step is not None and pretrain_step >= PRETRAIN_STEPS:
    print(f"Pretraining already completed: step {pretrain_step}/{PRETRAIN_STEPS}.")
else:
    command = [
        sys.executable, "-m", "qmini_addsub.train",
        "--train-bin", DATA_DIR / "pretrain_train.u16",
        "--valid-bin", DATA_DIR / "pretrain_validation.u16",
        "--tokenizer", TOKENIZER_PATH,
        "--config", MODEL_CONFIG_PATH,
        "--output", PRETRAIN_DIR,
        "--device", DEVICE,
        "--steps", str(PRETRAIN_STEPS),
        "--batch-size", str(PRETRAIN_BATCH_SIZE),
        "--sequence-length", str(PRETRAIN_SEQUENCE_LENGTH),
        "--save-every", str(PRETRAIN_SAVE_EVERY),
        "--validate-every", str(PRETRAIN_VALIDATE_EVERY),
        "--threads", str(CPU_THREADS),
    ]
    if pretrain_checkpoint is not None:
        print(f"Resuming safely from step {pretrain_step}: {pretrain_checkpoint}")
        command += ["--resume", pretrain_checkpoint]
    else:
        print("No previous pretraining checkpoint found; initializing the new model.")
    run(command, cwd=REPO_DIR)

pretrain_checkpoint, pretrain_step = resolve_checkpoint(
    PRETRAIN_DIR,
    prefix="checkpoint",
    metrics_filename="metrics.jsonl",
    explicit=None,
)
if pretrain_checkpoint is None or pretrain_step is None or pretrain_step < 1:
    raise RuntimeError("Pretraining did not produce a resumable non-zero-step checkpoint.")
if pretrain_step < PRETRAIN_STEPS:
    raise RuntimeError(
        f"Pretraining stopped at step {pretrain_step:,}/{PRETRAIN_STEPS:,}. "
        "The saved checkpoint is safe. Rerun this same cell to resume."
    )
print("Pretraining checkpoint:", pretrain_checkpoint)

print("\n[7/9] English assistant instruction fine-tuning (human-authored OASST1/Dolly)...")
sft_checkpoint, sft_step = resolve_checkpoint(
    SFT_DIR,
    prefix="sft-checkpoint",
    metrics_filename="sft_metrics.jsonl",
    explicit=None,
)
if sft_step is not None and sft_step >= SFT_STEPS:
    print(f"Assistant fine-tuning already completed: step {sft_step}/{SFT_STEPS}.")
else:
    command = [
        sys.executable, "-m", "qmini_addsub.sft",
        "--base-checkpoint", pretrain_checkpoint,
        "--tokenizer", TOKENIZER_PATH,
        "--train-jsonl", DATA_DIR / "sft_train.jsonl",
        "--valid-jsonl", DATA_DIR / "sft_validation.jsonl",
        "--test-jsonl", DATA_DIR / "sft_test.jsonl",
        "--output", SFT_DIR,
        "--device", DEVICE,
        "--threads", str(CPU_THREADS),
        "--steps", str(SFT_STEPS),
        "--batch-size", str(SFT_BATCH_SIZE),
        "--save-every", str(SFT_SAVE_EVERY),
    ]
    if sft_checkpoint is not None:
        print(f"Resuming assistant fine-tuning from step {sft_step}: {sft_checkpoint}")
        command += ["--resume", sft_checkpoint]
    else:
        print("Starting assistant fine-tuning from the completed pretraining checkpoint.")
    run(command, cwd=REPO_DIR)

sft_checkpoint, sft_step = resolve_checkpoint(
    SFT_DIR,
    prefix="sft-checkpoint",
    metrics_filename="sft_metrics.jsonl",
    explicit=None,
)
if sft_checkpoint is None or sft_step is None or sft_step < 1:
    raise RuntimeError("Assistant fine-tuning did not produce a non-zero-step checkpoint.")
if sft_step < SFT_STEPS:
    raise RuntimeError(
        f"Assistant fine-tuning stopped at step {sft_step:,}/{SFT_STEPS:,}. "
        "The saved checkpoint is safe. Rerun this same cell to resume."
    )
print("Assistant checkpoint:", sft_checkpoint)

print("\n[8/9] Held-out assistant evaluation and readiness report...")
checkpoint_sha = sha256_file(sft_checkpoint)
heldout_dir = EVALUATION_DIR / f"heldout-step-{sft_step:09d}-{checkpoint_sha[:12]}"
heldout_summary_path = heldout_dir / "heldout_summary.json"
if not heldout_summary_path.is_file():
    run([
        sys.executable, "-m", "qmini_addsub.evaluate",
        "--checkpoint", sft_checkpoint,
        "--tokenizer", TOKENIZER_PATH,
        "--test-jsonl", DATA_DIR / "sft_test.jsonl",
        "--dataset-manifest", MANIFEST_PATH,
        "--output", heldout_dir,
        "--device", DEVICE,
        "--max-new-tokens", "96",
    ], cwd=REPO_DIR)
else:
    heldout_summary = json.loads(heldout_summary_path.read_text(encoding="utf-8"))
    if heldout_summary.get("checkpoint_sha256") != checkpoint_sha:
        raise RuntimeError("Existing held-out report belongs to a different checkpoint.")

readiness_path = ARTIFACT_DIR / f"readiness-step-{sft_step:09d}-{checkpoint_sha[:12]}.json"
if not readiness_path.is_file():
    readiness_result = subprocess.run([
        sys.executable, "-m", "qmini_addsub.readiness",
        "--metrics", SFT_DIR / "sft_metrics.jsonl",
        "--checkpoint", sft_checkpoint,
        "--heldout-summary", heldout_summary_path,
        "--output", readiness_path,
    ], cwd=REPO_DIR, check=False)
    if readiness_result.returncode not in (0, 2) or not readiness_path.is_file():
        raise RuntimeError("Readiness gate failed to produce a report.")
readiness_report = json.loads(readiness_path.read_text(encoding="utf-8"))
print("Readiness:", readiness_report["status"])
print("Readiness checks:", f'{readiness_report["checks_passed"]}/{readiness_report["checks_total"]}')
if not readiness_report["ready"]:
    print("HUMAN REVIEW REQUIRED: do not present this checkpoint as a validated assistant.")

print("\n[9/9] Exporting the trained Add/Sub weight archive and saving the run summary...")
export_path = ARTIFACT_DIR / f"qmini_addsub_2m_sft-step-{sft_step:09d}-{checkpoint_sha[:12]}.qma"
if not export_path.is_file():
    run([
        sys.executable, "-m", "qmini_addsub.export",
        "--checkpoint", sft_checkpoint,
        "--output", export_path,
    ], cwd=REPO_DIR)
else:
    print("Packed Add/Sub archive already exists:", export_path)

summary = {
    "format": "qmini-addsub-one-click-run-v1",
    "repository_commit": REPO_COMMIT,
    "language": "English",
    "device": DEVICE,
    "cpu_threads": CPU_THREADS,
    "parameter_count": parameter_count,
    "parameter_budget": MAX_PARAMETER_COUNT,
    "vocab_size": model_config.vocab_size,
    "context_length": model_config.max_context,
    "pretraining_steps": pretrain_step,
    "pretraining_checkpoint": str(pretrain_checkpoint),
    "sft_steps": sft_step,
    "sft_checkpoint": str(sft_checkpoint),
    "sft_checkpoint_sha256": checkpoint_sha,
    "heldout_report": str(heldout_summary_path),
    "readiness_report": str(readiness_path),
    "readiness_status": readiness_report["status"],
    "packed_weight_archive": str(export_path),
    "dataset_manifest": str(MANIFEST_PATH),
    "synthetic_training_records_included": 0,
    "note": (
        "The .qma archive is a verified packed-weight container, not a completed "
        "integer-only/multiply-free mobile inference engine. Held-out lexical metrics "
        "require human review and do not certify factuality or reasoning."
    ),
}
summary_path = ARTIFACT_DIR / "training_summary.json"
summary_temp = summary_path.with_suffix(".json.tmp")
summary_temp.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
summary_temp.replace(summary_path)

print("\n" + "=" * 72)
print("Q-MINI ADD/SUB 2M — PIPELINE FINISHED")
print("=" * 72)
print("Parameters:", f"{parameter_count:,}", "/", f"{MAX_PARAMETER_COUNT:,}")
print("Pretraining:", f"{pretrain_step:,}", "steps")
print("Assistant SFT:", f"{sft_step:,}", "steps")
print("Readiness:", readiness_report["status"])
print("Training summary:", summary_path)
print("Packed weights:", export_path)
print("Drive workspace:", WORKSPACE_ROOT)
print("\nIf Colab disconnects before this message, rerun this same cell.")
print("It will verify and resume from the latest Drive checkpoint without resetting the run.")
